# Proyecto de Minería de Datos: del dato crudo al árbol de decisión

Este cuaderno corre **todo el proyecto paso a paso**, una celda a la vez, y genera cada gráfico.

**Qué vamos a hacer**

| Parte | Qué se hace | Fase CRISP-DM |
|---|---|---|
| 1 | Configurar y cargar los tres conjuntos de datos | Entender los datos |
| 2 | Revisar la calidad: vacíos, filas repetidas, categorías y valores extremos | Entender los datos |
| 3 | Limpiar los datos | Preparar los datos |
| 4 | Procesar por bloques (lotes) para archivos grandes | Preparar los datos |
| 5 | Gráficos exploratorios | Entender los datos |
| 6 | Escalar y codificar las variables | Preparar los datos |
| 7 | **Unir los tres conjuntos en una sola tabla** | Preparar los datos |
| 8 | **Un único árbol de decisión** sobre los tres conjuntos | Modelar |
| 9 | Conclusiones | Evaluar |

> **Sobre el árbol:** el proyecto construye **un solo árbol de decisión que junta los tres conjuntos de datos** (`adultos_salud`, `insurance` y `Medicalpremium`). Como cada uno predice algo distinto y tiene columnas distintas, la parte 7 explica paso a paso cómo se unen y qué decisiones hubo que tomar.

**Cómo usarlo:** ejecuta las celdas en orden con `Shift + Enter`. Este archivo debe estar en la raíz de `proyecto_mineria/` (junto a la carpeta `data/`).

## 1. Configuración y carga de datos

### 1.1 Importar las librerías
Cada librería tiene un trabajo: `pandas` maneja tablas, `matplotlib` y `seaborn` hacen gráficos, y `scikit-learn` tiene el árbol de decisión.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import ConfusionMatrixDisplay, f1_score, precision_score, recall_score

pd.set_option("display.max_columns", 30)     # mostrar todas las columnas en las tablas
sns.set_theme(style="whitegrid")             # estilo de los gráficos
print("Librerías cargadas")

Librerías cargadas


### 1.2 Definir las carpetas
`Path.cwd()` es la carpeta donde está este cuaderno. Desde ahí buscamos los datos y guardamos los gráficos.

In [2]:
RAIZ = Path.cwd()                                    # raíz del proyecto
RAW = RAIZ / "data" / "raw"                          # datos originales (sin limpiar)
LIMPIO = RAIZ / "data" / "limpio"                    # datos ya limpios
SALIDA = RAIZ / "data" / "cuaderno"                  # aquí guarda resultados este cuaderno
GRAFICAS = RAIZ / "graficas" / "cuaderno"            # aquí se guardan las imágenes

SALIDA.mkdir(parents=True, exist_ok=True)
GRAFICAS.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", RAIZ)
for archivo in [RAW / "adultos_salud.csv",
                LIMPIO / "insurance_limpio.csv",
                LIMPIO / "Medicalpremium_limpio.csv"]:
    print(f"{archivo.name:<28}", "OK" if archivo.exists() else "NO ENCONTRADO")

Raíz del proyecto: c:\Users\Felipe\Downloads\proyecto_mineria_v1.14\proyecto_mineriav14
adultos_salud.csv            NO ENCONTRADO
insurance_limpio.csv         NO ENCONTRADO
Medicalpremium_limpio.csv    NO ENCONTRADO


### 1.3 Cargar `adultos_salud` (archivo original, sin limpiar)
Este es el conjunto principal: 319.795 personas y si tienen o no enfermedad cardíaca (`HeartDisease`).

In [3]:
adultos = pd.read_csv(RAW / "adultos_salud.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Felipe\\Downloads\\proyecto_mineria_v1.14\\proyecto_mineriav14\\data\\raw\\adultos_salud.csv'

In [ ]:
print("Filas y columnas:", adultos.shape)

In [ ]:
adultos.head()

In [ ]:
adultos.dtypes

In [ ]:
memoria_total = adultos.memory_usage(deep=True).sum() / 1e6
print(f"Este archivo ocupa {memoria_total:.1f} MB en memoria")

### 1.4 Cargar `insurance` y `Medicalpremium` (ya limpios)
De estos dos conjuntos solo tenemos las versiones limpias.

In [ ]:
insurance = pd.read_csv(LIMPIO / "insurance_limpio.csv")
medical = pd.read_csv(LIMPIO / "Medicalpremium_limpio.csv")
print("insurance:", insurance.shape)
print("Medicalpremium:", medical.shape)

In [ ]:
insurance.head()

In [ ]:
medical.head()

### 1.5 Resumen de los tres conjuntos

In [ ]:
resumen = pd.DataFrame({
    "Datos": ["adultos_salud", "insurance", "Medicalpremium"],
    "Filas": [len(adultos), len(insurance), len(medical)],
    "Columnas": [adultos.shape[1], insurance.shape[1], medical.shape[1]],
    "Qué se predice": ["HeartDisease (Sí/No)", "charges (costo)", "PremiumPrice (prima)"],
    "Tipo de problema": ["Clasificación", "Regresión", "Regresión"],
})
resumen

**Clasificación** es elegir una categoría (¿tiene enfermedad: sí o no?). **Regresión** es predecir un número (¿cuánto costará?).

## 2. Revisar la calidad de los datos (`adultos_salud`)

Antes de limpiar hay que **mirar**. Son cuatro revisiones: vacíos, filas repetidas, categorías y valores extremos.

### 2.1 Valores vacíos (nulos)

In [ ]:
nulos = adultos.isna().sum()
nulos

In [ ]:
print("Total de valores vacíos:", nulos.sum())

### 2.2 Filas repetidas

In [ ]:
cantidad_repetidas = adultos.duplicated().sum()
porcentaje = cantidad_repetidas / len(adultos) * 100
print(f"Filas repetidas: {cantidad_repetidas} ({porcentaje:.1f} % del archivo)")

Ejemplo: estas son filas **idénticas en las 18 columnas** (se muestran ordenadas para ver las parejas).

In [ ]:
adultos[adultos.duplicated(keep=False)].sort_values(list(adultos.columns)).head(6)

### 2.3 Categorías
Revisamos que el texto sea consistente (por ejemplo, que no exista `yes`, `YES` y `Yes` a la vez).

In [ ]:
columnas_texto = adultos.select_dtypes(exclude="number").columns.tolist()
for columna in columnas_texto:
    print(columna, "->", sorted(adultos[columna].unique()))

Todas las categorías son consistentes, así que **no hace falta corregir textos**.

### 2.4 Valores extremos (outliers) con la regla IQR
Un valor es extremo si se sale del rango normal: de `Q1 - 1,5 × IQR` a `Q3 + 1,5 × IQR`. Solo los **contamos**; no los borramos.

In [ ]:
columnas_numericas = adultos.select_dtypes(include="number").columns.tolist()
print("Columnas numéricas:", columnas_numericas)

In [ ]:
filas = []
for columna in columnas_numericas:
    q1, q3 = adultos[columna].quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    extremos = adultos[(adultos[columna] < limite_inferior) | (adultos[columna] > limite_superior)]
    filas.append({"Variable": columna, "Q1": q1, "Q3": q3, "IQR": iqr,
                  "Límite inferior": limite_inferior, "Límite superior": limite_superior,
                  "Valores extremos": len(extremos)})
tabla_iqr = pd.DataFrame(filas)
tabla_iqr

Se **conservan**: son casos reales (por ejemplo, personas con obesidad severa o muchos días de mala salud), no errores de captura.

## 3. Limpiar los datos

La única regla que cambia los datos es **eliminar filas repetidas** (se deja la primera aparición).

In [ ]:
print("Antes :", len(adultos), "filas")
adultos_limpio = adultos.drop_duplicates().reset_index(drop=True)
print("Después:", len(adultos_limpio), "filas")
print("Eliminadas:", len(adultos) - len(adultos_limpio))

In [ ]:
print("Nulos restantes:", adultos_limpio.isna().sum().sum())
print("Repetidas restantes:", adultos_limpio.duplicated().sum())

In [ ]:
adultos_limpio.to_csv(SALIDA / "adultos_salud_limpio.csv", index=False)
print("Guardado en:", SALIDA / "adultos_salud_limpio.csv")

### 3.1 Comprobar que `insurance` y `Medicalpremium` también están limpios

In [ ]:
for nombre, tabla in [("insurance", insurance), ("Medicalpremium", medical)]:
    print(f"{nombre}: nulos = {tabla.isna().sum().sum()} | repetidas = {tabla.duplicated().sum()}")

## 4. Escalabilidad: procesar por bloques (lotes)

**Problema:** si el archivo es enorme, cargarlo completo en memoria puede ser imposible.
**Solución:** leerlo en **bloques de filas**, procesar cada bloque y guardarlo antes de leer el siguiente.

### 4.1 La idea: un bloque pesa mucho menos que el archivo completo

In [ ]:
lector = pd.read_csv(RAW / "adultos_salud.csv", chunksize=10000)   # NO carga todo: crea un lector
bloque = next(lector)                                              # pide solo el primer bloque
memoria_bloque = bloque.memory_usage(deep=True).sum() / 1e6
print(f"Archivo completo: {memoria_total:.1f} MB | un bloque de 10.000 filas: {memoria_bloque:.1f} MB")

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(["Archivo completo", "Un bloque\n(10.000 filas)"], [memoria_total, memoria_bloque],
        color=["#e45756", "#4c78a8"])
plt.ylabel("Memoria (MB)")
plt.title("Cargar todo vs cargar por bloques")
plt.tight_layout()
plt.savefig(GRAFICAS / "1_memoria_bloque.png", dpi=130)
plt.show()

### 4.2 Recorrer el archivo bloque por bloque

In [ ]:
filas_leidas = 0
for numero, bloque in enumerate(pd.read_csv(RAW / "adultos_salud.csv", chunksize=50000), start=1):
    filas_leidas += len(bloque)
    print(f"Bloque {numero}: {len(bloque):>6} filas | acumulado: {filas_leidas}")

### 4.3 Limpieza completa por bloques (clase `BatchCleaner` del proyecto)
Hace lo mismo que la parte 3, pero sin cargar todo el archivo. Lo difícil: una fila puede repetirse en **bloques distintos**, así que guarda una "huella" de cada fila ya escrita para detectarlo.

In [ ]:
from limpieza.lotes_cleaner import BatchCleaner

limpiador = BatchCleaner(
    ruta_raw=RAW / "adultos_salud.csv",
    ruta_limpio=SALIDA / "adultos_salud_por_lotes.csv",
    nombre_dataset="adultos_salud.csv",
    columnas_a_verificar=columnas_texto,
    columnas_iqr=columnas_numericas,
    tamano_lote=10000,
)
registro = limpiador.ejecutar()

### 4.4 ¿Da el mismo resultado que la limpieza normal?

In [ ]:
adultos_por_lotes = pd.read_csv(SALIDA / "adultos_salud_por_lotes.csv")
print("Filas (normal):", len(adultos_limpio), "| Filas (por bloques):", len(adultos_por_lotes))
print("¿Tablas idénticas?:", adultos_limpio.equals(adultos_por_lotes))

**Medición real** hecha con `benchmark_lotes.py` (bloques de 10.000 filas, sobre este mismo archivo):

| | Normal | Por bloques |
|---|---|---|
| Memoria extra al limpiar | 110 MB | 36 MB |
| Memoria extra al preparar | 157 MB | 17 MB |
| Tiempo al limpiar | 2,4 s | 4,5 s |

Se gana memoria y se pierde algo de tiempo. Con un archivo de 25 MB no hace falta, pero con archivos mucho más grandes sí.

## 5. Gráficos exploratorios

### 5.1 ¿Cuántas personas tienen la enfermedad?

In [ ]:
conteo = adultos_limpio["HeartDisease"].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(["Sin enfermedad", "Con enfermedad"], [conteo["No"], conteo["Yes"]], color=["#4c78a8", "#e45756"])
for posicion, valor in enumerate([conteo["No"], conteo["Yes"]]):
    plt.text(posicion, valor, f"{valor / len(adultos_limpio):.1%}", ha="center", va="bottom")
plt.ylabel("Cantidad de personas")
plt.title("Muy pocas personas tienen la enfermedad (clases desbalanceadas)")
plt.tight_layout()
plt.savefig(GRAFICAS / "2_distribucion_objetivo.png", dpi=130)
plt.show()

Esto es clave: **9 de cada 100 personas** tienen la enfermedad. Más adelante veremos por qué importa.

### 5.2 Distribución de las variables numéricas

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(12, 8))
for eje, columna in zip(ejes.ravel(), columnas_numericas):
    eje.hist(adultos_limpio[columna], bins=30, color="#4c78a8")
    eje.set_title(f"Distribución de {columna}")
    eje.set_xlabel(columna)
    eje.set_ylabel("Cantidad de personas")
plt.tight_layout()
plt.savefig(GRAFICAS / "3_histogramas.png", dpi=130)
plt.show()

### 5.3 Valores extremos (diagramas de caja)
Los puntos fuera de la caja son los valores extremos que contamos en la parte 2.4.

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(15, 4))
for eje, columna in zip(ejes, columnas_numericas):
    sns.boxplot(y=adultos_limpio[columna], ax=eje, color="#9ecae9")
    eje.set_title(columna)
    eje.set_ylabel("")
plt.tight_layout()
plt.savefig(GRAFICAS / "4_cajas_extremos.png", dpi=130)
plt.show()

### 5.4 ¿Qué variables se relacionan con la enfermedad?
Para cada variable, el porcentaje de personas con enfermedad en cada grupo. Cuanto más distintas son las barras, más separa esa variable a los enfermos.

In [ ]:
variables = ["DiffWalking", "Stroke", "Diabetic", "GenHealth", "Smoking", "Sex"]
fig, ejes = plt.subplots(2, 3, figsize=(17, 9))
for eje, variable in zip(ejes.ravel(), variables):
    porcentaje = adultos_limpio.groupby(variable)["HeartDisease"].apply(lambda s: (s == "Yes").mean() * 100)
    porcentaje.sort_values().plot(kind="barh", ax=eje, color="#4c78a8")
    eje.set_title(f"% con enfermedad según {variable}")
    eje.set_xlabel("% con enfermedad cardíaca")
    eje.set_ylabel("")
plt.tight_layout()
plt.savefig(GRAFICAS / "5_enfermedad_por_variable.png", dpi=130)
plt.show()

Quienes tienen dificultad para caminar, antecedente de ACV (`Stroke`) o diabetes tienen un porcentaje de enfermedad mucho más alto. Son señales de riesgo claras. En la parte 7 veremos cuáles de ellas se pueden usar en el árbol único.

### 5.5 Relación entre las variables numéricas

In [ ]:
numericas_con_objetivo = adultos_limpio[columnas_numericas].copy()
numericas_con_objetivo["HeartDisease (1 = sí)"] = (adultos_limpio["HeartDisease"] == "Yes").astype(int)
correlacion = numericas_con_objetivo.corr()

plt.figure(figsize=(7, 5))
sns.heatmap(correlacion, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlación (cercano a 1 o -1 = relación fuerte)")
plt.tight_layout()
plt.savefig(GRAFICAS / "6_correlacion.png", dpi=130)
plt.show()

### 5.6 Los otros dos conjuntos: `insurance` y `Medicalpremium`

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=insurance, x="smoker", y="charges", ax=ejes[0], color="#9ecae9")
ejes[0].set_title("insurance: costo médico según si fuma")
ejes[0].set_xlabel("¿Fuma? (no / yes)")
ejes[0].set_ylabel("Costo médico (charges)")

ejes[1].scatter(medical["Age"], medical["PremiumPrice"], s=12, alpha=0.5)
ejes[1].set_title("Medicalpremium: precio de la prima según la edad")
ejes[1].set_xlabel("Edad")
ejes[1].set_ylabel("Precio de la prima")

plt.tight_layout()
plt.savefig(GRAFICAS / "7_insurance_medicalpremium.png", dpi=130)
plt.show()

Fumar dispara el costo en `insurance`, y la edad sube el precio de la prima en `Medicalpremium`.

## 6. Preparar las variables: escalar y codificar

### 6.1 Escalar las variables numéricas (`StandardScaler`)
Las variables numéricas tienen escalas distintas (el IMC va de 12 a 95, los días de mala salud de 0 a 30). Escalar las lleva a una **escala común**: promedio 0 y desviación 1.

> **Evitar la fuga de datos (*data leakage*).** El escalador aprende un promedio y una desviación. Si los aprendiera con **todas** las personas, la información de las personas de prueba se "filtraría" al entrenamiento y la evaluación saldría más optimista que la realidad. Por eso aquí primero se **separan** entrenamiento (75 %) y prueba (25 %), el escalador se **ajusta solo con entrenamiento** (`fit`) y después se aplica a ambos (`transform`). En el modelado real (parte 9) esto se automatiza con un `Pipeline` de scikit-learn, que repite el mismo orden en cada entrenamiento.

In [ ]:
print("ANTES de escalar:")
adultos_limpio[columnas_numericas].describe().loc[["mean", "std", "min", "max"]].round(2)

In [ ]:
adultos_train, adultos_test = train_test_split(
    adultos_limpio, test_size=0.25, random_state=42, stratify=adultos_limpio["HeartDisease"])

escalador = StandardScaler().fit(adultos_train[columnas_numericas])      # aprende SOLO con entrenamiento
train_escalado = pd.DataFrame(escalador.transform(adultos_train[columnas_numericas]),
                              columns=columnas_numericas, index=adultos_train.index)
test_escalado = pd.DataFrame(escalador.transform(adultos_test[columnas_numericas]),
                             columns=columnas_numericas, index=adultos_test.index)

print("Entrenamiento (promedio 0 y desviación 1 por construcción):")
display(train_escalado.describe().loc[["mean", "std"]].round(3))
print("Prueba (queda cerca de 0 y 1, pero no exacto: el escalador no vio estos datos):")
display(test_escalado.describe().loc[["mean", "std"]].round(3))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].hist(adultos_train["BMI"], bins=40, color="#4c78a8")
ejes[0].set_title("BMI antes de escalar (entrenamiento)")
ejes[1].hist(train_escalado["BMI"], bins=40, color="#f58518")
ejes[1].set_title("BMI después de escalar (promedio 0)")
for eje in ejes:
    eje.set_ylabel("Cantidad de personas")
plt.tight_layout()
plt.savefig(GRAFICAS / "8_antes_despues_escalado.png", dpi=130)
plt.show()

La **forma** de la distribución es la misma; solo cambió la escala del eje.

### 6.2 Convertir texto en números (one-hot)
El árbol solo entiende números. Cada categoría pasa a ser una columna de 0 y 1.

In [ ]:
categoricas = [c for c in adultos_limpio.columns if c not in columnas_numericas and c != "HeartDisease"]
print("Variables de texto:", len(categoricas))
print(categoricas)

In [ ]:
codificadas = pd.get_dummies(adultos_limpio[categoricas], drop_first=True, dtype=int)
print("Columnas nuevas:", codificadas.shape[1])
codificadas.iloc[:, :6].head()

### 6.3 Guardar la matriz escalada y codificada
Queda lista para modelos que **sí** necesitan escalar (KNN, regresión lineal). La columna `conjunto` indica qué filas son de entrenamiento y cuáles de prueba: la matriz se escaló con los valores aprendidos **solo** del entrenamiento. (El *one-hot* no aprende nada de los datos más allá de la lista fija de categorías, así que no filtra información.)

In [ ]:
numericas_escaladas = pd.concat([train_escalado, test_escalado])
matriz_escalada = pd.concat([numericas_escaladas, codificadas, adultos_limpio[["HeartDisease"]]], axis=1)
matriz_escalada["conjunto"] = np.where(matriz_escalada.index.isin(adultos_train.index), "entrenamiento", "prueba")
matriz_escalada.to_csv(SALIDA / "adultos_salud_escalado.csv", index=False)
print("Forma de la matriz:", matriz_escalada.shape)
print(matriz_escalada["conjunto"].value_counts().to_dict())

> **Dato clave:** el árbol de decisión **no necesita escalar**, porque solo compara contra un valor (por ejemplo, ¿IMC ≤ 30?) y eso no cambia con la escala. Por eso el árbol de abajo usa las variables **sin escalar**. Los modelos que sí lo necesitan (KNN, K-means) usan el escalador dentro de un `Pipeline` (parte 9) o, en el clustering de la parte 10, sobre todos los datos porque ahí no hay partición de prueba que proteger.

## 7. Unir los tres conjuntos de datos

Hasta aquí cada conjunto se trabajó por separado. Ahora se unen para construir **un solo árbol**. Hay tres problemas:

1. **Columnas distintas:** cada conjunto tiene variables diferentes.
2. **Objetivos distintos:** uno predice una enfermedad (Sí/No) y los otros dos predicen un costo.
3. **Tamaños muy distintos:** 301.717, 500.000 y 986 filas.

Cada problema se resuelve con una decisión, y se explican una por una.

### 7.1 Decisión 1: usar solo las variables que se comparten
Una variable que existe en un solo conjunto (por ejemplo `children` o `Stroke`) le serviría al árbol únicamente para **adivinar de qué conjunto viene la fila**. Por eso solo se usan variables que existen, o se pueden calcular, en **al menos dos** conjuntos:

| Variable común | adultos_salud | insurance | Medicalpremium |
|---|---|---|---|
| `edad` | `AgeCategory` (rangos, se usa el punto medio) | `age` | `Age` |
| `imc` | `BMI` | `bmi` | se calcula: `Weight / (Height/100)²` |
| `hombre` (1 = sí) | `Sex` | `sex` | no disponible |
| `fuma` (1 = sí) | `Smoking` | `smoker` | no disponible |
| `diabetes` (1 = sí) | `Diabetic` | no disponible | `Diabetes` |

Cuando una variable **no existe** en un conjunto se escribe `-1` ("dato no disponible").

### 7.2 Decisión 2: un objetivo común, `riesgo_alto`
Cada conjunto define el riesgo alto a su manera: tener enfermedad cardíaca (`adultos_salud`), un costo médico muy alto (`insurance`) o una prima muy alta (`Medicalpremium`). Para que sea comparable, en los tres se marca como riesgo alto **el mismo porcentaje de personas: el 9 %**, que es el porcentaje que tiene la enfermedad en `adultos_salud`.

*¿Por qué el mismo porcentaje?* Si en un conjunto el riesgo alto fuera el 25 % y en otro el 9 %, el árbol acertaría más solo adivinando el conjunto de origen, y eso no sería aprender nada útil.

In [ ]:
tasa_riesgo = (adultos_limpio["HeartDisease"] == "Yes").mean()
print(f"Porcentaje de riesgo alto en los tres conjuntos: {tasa_riesgo:.1%}")

In [ ]:
# Un costo o una prima cuenta como "riesgo alto" si está entre el 9 % más alto de su conjunto
umbral_costo = insurance["charges"].quantile(1 - tasa_riesgo)
umbral_prima = medical["PremiumPrice"].quantile(1 - tasa_riesgo)
print(f"insurance: riesgo alto si charges >= {umbral_costo:,.0f}")
print(f"Medicalpremium: riesgo alto si PremiumPrice >= {umbral_prima:,.0f}")

### 7.3 Armonizar `adultos_salud`
Primero la edad: viene en rangos como `18-24`. Se reemplaza cada rango por su **punto medio** (21, 27, 32...). Es una aproximación.

In [ ]:
punto_medio = {}
for rango in adultos_limpio["AgeCategory"].unique():
    if rango == "80 or older":
        punto_medio[rango] = 82                       # supuesto para el último grupo
    else:
        punto_medio[rango] = (int(rango[:2]) + int(rango[3:5])) / 2
dict(sorted(punto_medio.items()))

In [ ]:
t_adultos = pd.DataFrame({
    "origen": "adultos_salud",
    "edad": adultos_limpio["AgeCategory"].map(punto_medio),
    "imc": adultos_limpio["BMI"],
    "hombre": (adultos_limpio["Sex"] == "Male").astype(int),
    "fuma": (adultos_limpio["Smoking"] == "Yes").astype(int),
    "diabetes": adultos_limpio["Diabetic"].str.startswith("Yes").astype(int),
    "riesgo_alto": (adultos_limpio["HeartDisease"] == "Yes").astype(int),
})
t_adultos.head()

### 7.4 Armonizar `insurance`
No tiene diabetes, así que esa columna queda en `-1`.

In [ ]:
t_insurance = pd.DataFrame({
    "origen": "insurance",
    "edad": insurance["age"],
    "imc": insurance["bmi"],
    "hombre": (insurance["sex"] == "male").astype(int),
    "fuma": (insurance["smoker"] == "yes").astype(int),
    "diabetes": -1,                                    # no disponible
    "riesgo_alto": (insurance["charges"] >= umbral_costo).astype(int),
})
t_insurance.head()

### 7.5 Armonizar `Medicalpremium`
No tiene sexo ni fumar (quedan en `-1`) y el IMC se calcula con el peso y la estatura.

In [ ]:
t_medical = pd.DataFrame({
    "origen": "Medicalpremium",
    "edad": medical["Age"],
    "imc": medical["Weight"] / (medical["Height"] / 100) ** 2,   # peso (kg) / estatura (m) al cuadrado
    "hombre": -1,                                      # no disponible
    "fuma": -1,                                        # no disponible
    "diabetes": medical["Diabetes"],
    "riesgo_alto": (medical["PremiumPrice"] >= umbral_prima).astype(int),
})
t_medical.head()

In [ ]:
for nombre, tabla_origen in [("adultos_salud", t_adultos), ("insurance", t_insurance), ("Medicalpremium", t_medical)]:
    print(f"{nombre:<15} riesgo alto: {tabla_origen['riesgo_alto'].mean():.1%} de {len(tabla_origen)} filas")

Los tres conjuntos quedan con el mismo porcentaje de riesgo alto (9,0 %).

### 7.6 Decisión 3: una muestra del mismo tamaño de cada conjunto
`insurance` tiene 500.000 filas y `Medicalpremium` solo 986. Si se juntaran tal cual, el árbol aprendería casi solo `insurance`. Por eso se toman **986 filas al azar de cada conjunto** (el tamaño del más pequeño). `random_state=42` hace que la muestra sea siempre la misma.

In [ ]:
n = min(len(t_adultos), len(t_insurance), len(t_medical))
print("Filas por conjunto:", n)

In [ ]:
tabla = pd.concat(
    [t.sample(n=n, random_state=42) for t in (t_adultos, t_insurance, t_medical)],
    ignore_index=True)
print("Tabla unificada:", tabla.shape)
tabla.groupby("origen")["riesgo_alto"].agg(filas="count", riesgo_alto="sum")

In [ ]:
resumen_unido = tabla.groupby(["origen", "riesgo_alto"]).size().unstack(fill_value=0)
resumen_unido.columns = ["Riesgo normal", "Riesgo alto"]
resumen_unido.plot(kind="bar", stacked=True, figsize=(8, 5), color=["#f58518", "#4c78a8"])
plt.title("Tabla unificada: la misma cantidad de filas de cada conjunto")
plt.xlabel("Conjunto de origen")
plt.ylabel("Cantidad de personas")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(GRAFICAS / "13_tabla_unificada.png", dpi=130)
plt.show()

In [ ]:
tabla.to_csv(SALIDA / "tabla_unificada.csv", index=False)
print("Guardada en:", SALIDA / "tabla_unificada.csv")
tabla.sample(8, random_state=1)

## 8. El árbol de decisión único

Pregunta que responde: **¿esta persona es de riesgo alto?** (enfermedad cardíaca, costo médico muy alto o prima muy alta, según el conjunto de donde venga).

### 8.1 Preparar X (características) e y (lo que se predice)

In [ ]:
variables = ["edad", "imc", "hombre", "fuma", "diabetes"]
X = tabla[variables]
y = tabla["riesgo_alto"]
print("Variables de entrada:", variables)
print("Personas:", len(X), "| con riesgo alto:", y.sum())

### 8.2 Separar datos de entrenamiento y de prueba
El árbol **aprende** con el 75 % y se **evalúa** con el 25 % que nunca vio. Se separa por grupos (`conjunto + riesgo`) para que los tres conjuntos queden bien representados en ambos lados.

In [ ]:
grupo = tabla["origen"] + "_" + y.astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=grupo)
print("Entrenamiento:", len(X_train), "personas | Prueba:", len(X_test), "personas")

### 8.3 Elegir cuántas preguntas seguidas hace el árbol (profundidad)
Se prueban varias profundidades **solo para elegir**, y solo con los datos de entrenamiento (validación cruzada: el entrenamiento se parte en 5 y se prueba 5 veces). Se mide con **F1**, que combina detectar el riesgo alto con no dar falsas alarmas. El árbol del proyecto es uno solo y se entrena después.

In [ ]:
validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = []
for profundidad in [2, 3, 4, 5, 6, 8]:
    ensayo = DecisionTreeClassifier(max_depth=profundidad, random_state=42, class_weight="balanced")
    puntajes = cross_val_score(ensayo, X_train, y_train, cv=validacion, scoring="f1")
    resultados.append({"Profundidad": profundidad, "F1 promedio": puntajes.mean(), "Variación": puntajes.std()})
tabla_profundidad = pd.DataFrame(resultados)
tabla_profundidad.round(3)

In [ ]:
plt.figure(figsize=(8, 5))
plt.errorbar(tabla_profundidad["Profundidad"], tabla_profundidad["F1 promedio"],
             yerr=tabla_profundidad["Variación"], marker="o", linewidth=2.5, capsize=4)
plt.xlabel("Profundidad del árbol (cantidad de preguntas seguidas)")
plt.ylabel("F1 en validación cruzada (más alto = mejor)")
plt.title("¿Cuántas preguntas seguidas conviene hacer?")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(GRAFICAS / "14_elegir_profundidad.png", dpi=130)
plt.show()

Con **4 preguntas seguidas** el F1 es el más alto, y todavía se puede leer el árbol completo. Más profundidad no mejora y haría el árbol ilegible.

`class_weight="balanced"` le pide al árbol que preste **la misma atención** al riesgo alto que al normal. Sin eso, como el riesgo alto es solo el 9 %, el árbol casi nunca lo detectaría.

### 8.4 Entrenar el árbol final

In [ ]:
arbol = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced")

In [ ]:
arbol.fit(X_train, y_train)

In [ ]:
prediccion = arbol.predict(X_test)
print(f"Aciertos en entrenamiento: {arbol.score(X_train, y_train):.3f}")
print(f"Aciertos en prueba       : {arbol.score(X_test, y_test):.3f}")
print(f"Riesgo alto detectado    : {recall_score(y_test, prediccion):.1%}")
print(f"Precisión (de los que marca como riesgo alto, cuántos lo son): {precision_score(y_test, prediccion):.1%}")
print(f"F1                       : {f1_score(y_test, prediccion):.3f}")

### 8.5 Dibujar el árbol
Se lee de arriba hacia abajo. Cada caja es una pregunta de sí o no. **Naranja = riesgo normal, azul = riesgo alto.** El número `value` es el peso de cada clase: como se pidió dar la misma atención a ambas, los pesos no son cantidades de personas.

In [ ]:
plt.figure(figsize=(26, 11))
plot_tree(arbol, feature_names=variables, class_names=["Riesgo normal", "Riesgo alto"],
          filled=True, rounded=True, fontsize=9, precision=1)
plt.title("Árbol de decisión único del proyecto (4 preguntas seguidas)")
plt.savefig(GRAFICAS / "15_arbol_unico.png", dpi=130, bbox_inches="tight")
plt.show()

### 8.6 Las reglas del árbol en texto
`-1` significa "dato no disponible". Por ejemplo, `diabetes <= -0.5` equivale a "este conjunto no tiene el dato de diabetes", es decir, la fila viene de `insurance`.

In [ ]:
print(export_text(arbol, feature_names=variables))

### 8.7 Qué variables usó el árbol

In [ ]:
importancia = pd.Series(arbol.feature_importances_, index=variables).sort_values()

plt.figure(figsize=(8, 4))
plt.barh(importancia.index, importancia.values, color="#4c78a8")
plt.xlabel("Importancia (qué tanto usó el árbol esta variable)")
plt.title("Variables que usó el árbol")
plt.tight_layout()
plt.savefig(GRAFICAS / "16_importancia.png", dpi=130)
plt.show()

In [ ]:
importancia.sort_values(ascending=False).round(3)

Que una variable pese mucho **no prueba que sea la causa** del riesgo: solo dice que ayuda a separar a las personas.

### 8.8 ¿Se equivoca? Matriz de confusión
Filas = lo que realmente ocurrió; columnas = lo que predijo el árbol. Cada fila suma 100 %.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, prediccion, normalize="true", values_format=".0%", colorbar=False,
    display_labels=["Riesgo normal", "Riesgo alto"])
plt.xlabel("Lo que predijo el árbol")
plt.ylabel("Lo que realmente ocurrió")
plt.title("Matriz de confusión (datos de prueba)")
plt.savefig(GRAFICAS / "17_matriz_confusion.png", dpi=130, bbox_inches="tight")
plt.show()

### 8.9 ¿Funciona igual en los tres conjuntos?
Esta es la comprobación más importante de un árbol que junta tres conjuntos distintos: medir **por separado** qué tan bien le va a cada uno.

In [ ]:
origen_prueba = tabla.loc[X_test.index, "origen"]
filas = []
for nombre in ["adultos_salud", "insurance", "Medicalpremium"]:
    mascara = origen_prueba == nombre
    filas.append({
        "Conjunto": nombre,
        "Personas en la prueba": int(mascara.sum()),
        "Casos de riesgo alto": int(y_test[mascara].sum()),
        "Riesgo alto detectado": recall_score(y_test[mascara], prediccion[mascara]),
        "Aciertos totales": (y_test[mascara] == prediccion[mascara]).mean(),
    })
por_conjunto = pd.DataFrame(filas).set_index("Conjunto")
por_conjunto.round(3)

In [ ]:
por_conjunto[["Riesgo alto detectado", "Aciertos totales"]].plot(
    kind="bar", figsize=(8, 5), color=["#e45756", "#4c78a8"])
plt.ylim(0, 1.05)
plt.title("¿El árbol funciona igual en los tres conjuntos?")
plt.ylabel("Proporción")
plt.xlabel("")
plt.xticks(rotation=0)
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig(GRAFICAS / "18_por_conjunto.png", dpi=130)
plt.show()

### 8.9b La limitación del `-1`: el árbol sabe de qué conjunto viene cada fila
Cuando un conjunto no tiene una variable se escribió `-1`. El problema: **cada conjunto tiene un patrón distinto de `-1`**, así que el `-1` funciona como una etiqueta del conjunto de origen. Se comprueba con un experimento: un árbol que **solo** ve dónde hay `-1` (sin edad, IMC ni nada más) intenta adivinar de qué conjunto viene la persona.

In [ ]:
faltantes = (X == -1).astype(int).add_suffix("_falta")
print("Patrón de -1 en cada conjunto (1 = la variable no existe en ese conjunto):")
display(faltantes.assign(origen=tabla["origen"]).drop_duplicates().sort_values("origen").set_index("origen"))

detective = DecisionTreeClassifier(max_depth=2, random_state=42).fit(faltantes, tabla["origen"])
print(f"Aciertos adivinando el conjunto de origen solo con los -1: {detective.score(faltantes, tabla['origen']):.0%}")

**Cómo explicarlo en voz alta:**

1. `insurance` no tiene diabetes → `diabetes = -1`. `Medicalpremium` no tiene sexo ni fumar → `hombre = -1` y `fuma = -1`. `adultos_salud` no tiene ninguno. Cada patrón es único.
2. El árbol puede hacer la pregunta `diabetes <= -0.5` y, con ella, separar `insurance` del resto. Desde ahí **aplica reglas distintas a cada conjunto**: en la práctica son tres modelos pegados en uno, no un modelo único de "riesgo".
3. Esto explica por qué el árbol detecta el 100 % del riesgo alto en `insurance` y mucho menos en los otros: aprendió la regla fácil de `insurance` (IMC alto y fumar) y la reservó para sus filas.
4. Además, `-1` es un número falso: una pregunta como `fuma <= 0.5` mezcla "no fuma" (0) con "no se sabe" (-1).
5. **Es la contrapartida de unir conjuntos con columnas distintas.** Alternativas: usar solo variables que existan en los tres (queda casi nada), o entrenar un modelo por conjunto (como en la versión 1.2 del proyecto).

### 8.10 Usar el árbol con una persona de cada conjunto
Para cada conjunto, una persona de prueba que sí es de riesgo alto.

In [ ]:
for nombre in ["adultos_salud", "insurance", "Medicalpremium"]:
    indice = y_test[(origen_prueba == nombre) & (y_test == 1)].index[0]
    persona = X_test.loc[[indice]]
    probabilidad = arbol.predict_proba(persona)[0, 1]
    veredicto = "riesgo alto" if arbol.predict(persona)[0] == 1 else "riesgo normal"
    print(f"{nombre:<15} | realidad: riesgo alto | el árbol dice: {veredicto:<13} | probabilidad {probabilidad:.0%}")
    print("   datos:", persona.round(1).iloc[0].to_dict())

### 8.11 ¿Qué tan estable es el resultado?
El resultado de arriba salió de **una sola muestra** (`random_state=42`) y de una prueba de solo 740 personas. Para saber cuánto cambia, se repite el proceso completo (muestra, separación y entrenamiento) con **20 semillas distintas**. *No son árboles nuevos del proyecto: es el mismo árbol repetido con otras muestras para medir cuánto varían los resultados.*

In [ ]:
repeticiones = []
for semilla in range(20):
    tabla_rep = pd.concat([t.sample(n=n, random_state=semilla) for t in (t_adultos, t_insurance, t_medical)],
                          ignore_index=True)
    X_r, y_r = tabla_rep[variables], tabla_rep["riesgo_alto"]
    grupo_r = tabla_rep["origen"] + "_" + y_r.astype(str)
    X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.25, random_state=semilla, stratify=grupo_r)
    arbol_r = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced").fit(X_tr, y_tr)
    pred_r = arbol_r.predict(X_te)
    origen_r = tabla_rep.loc[X_te.index, "origen"]
    fila = {"aciertos": arbol_r.score(X_te, y_te),
            "riesgo_alto_detectado": recall_score(y_te, pred_r),
            "precision": precision_score(y_te, pred_r, zero_division=0),
            "f1": f1_score(y_te, pred_r)}
    for nombre in ["adultos_salud", "insurance", "Medicalpremium"]:
        mascara = origen_r == nombre
        fila["detectado_" + nombre] = recall_score(y_te[mascara], pred_r[mascara])
    repeticiones.append(fila)
estabilidad = pd.DataFrame(repeticiones)
estabilidad.agg(["mean", "std", "min", "max"]).T.round(3)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 5))
estabilidad[["riesgo_alto_detectado", "precision", "f1"]].rename(columns={
    "riesgo_alto_detectado": "Riesgo alto\ndetectado", "precision": "Precisión", "f1": "F1"}).plot(kind="box", ax=ejes[0])
ejes[0].set_title("20 repeticiones: ¿cuánto varía el resultado?")
ejes[0].set_ylabel("Valor")
estabilidad[["detectado_adultos_salud", "detectado_insurance", "detectado_Medicalpremium"]].rename(columns={
    "detectado_adultos_salud": "adultos_salud", "detectado_insurance": "insurance",
    "detectado_Medicalpremium": "Medicalpremium"}).plot(kind="box", ax=ejes[1])
ejes[1].set_title("Riesgo alto detectado, por conjunto")
ejes[1].set_ylabel("Proporción detectada")
plt.tight_layout()
plt.savefig(GRAFICAS / "27_estabilidad.png", dpi=130)
plt.show()

**Lectura:** el árbol detecta en promedio el **75 %** del riesgo alto, pero según la muestra puede ir del 57 % al 85 %. La muestra con semilla 42 (61,5 %) fue de las más bajas, así que **conviene citar el rango y no un solo número**. Por conjunto la diferencia es grande: casi siempre acierta en `insurance`, y en `adultos_salud` y `Medicalpremium` varía mucho de una muestra a otra.

## 9. Dos rivales para el árbol: Naive Bayes y KNN

El árbol no es el único modelo posible. Aquí compiten **tres clasificadores** sobre **exactamente los mismos datos y la misma partición** de la parte 8 (`X_train`, `X_test`, `y_train`, `y_test`). Sin esa igualdad, la comparación no vale nada.

| Modelo | Idea en una frase | ¿Necesita escalar? |
|---|---|---|
| **Árbol** | Hace preguntas encadenadas de sí/no | No (compara contra umbrales) |
| **Naive Bayes** | Combina probabilidades suponiendo que las variables son independientes entre sí | No (modela cada variable aparte) |
| **KNN** | Guarda todos los casos y, ante uno nuevo, deja votar a sus *k* vecinos más parecidos | **Sí** (usa distancias) |

KNN es el modelo que usará la matriz escalada de la parte 6: *«distancia manda a escalar»*.

In [ ]:
import time
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

### 9.1 El torneo de tres modelos
Dos detalles de diseño:

- **KNN va dentro de un `Pipeline`** con el `StandardScaler`: así el escalador aprende solo con los datos de entrenamiento y después transforma los de prueba. Escalar antes de separar sería una fuga de datos.
- **Naive Bayes aparece dos veces.** Como solo el 9 % de las personas es de riesgo alto, con su configuración normal casi nunca "se anima" a predecir riesgo alto. La segunda versión le da un *a priori* de 50/50, que es el equivalente a lo que hace `class_weight="balanced"` en el árbol. Así la comparación es justa.

In [ ]:
modelos = {
    "Árbol (prof=4, balanced)": DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "Naive Bayes (priors 50/50)": GaussianNB(priors=[0.5, 0.5]),
    "KNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}

print(f"Un modelo que siempre dijera 'riesgo normal' acertaría {1 - y_test.mean():.3f}  <- hay que superar esto")

filas = []
for nombre, modelo in modelos.items():
    inicio = time.perf_counter()
    modelo.fit(X_train, y_train)
    segundos = time.perf_counter() - inicio
    pred = modelo.predict(X_test)
    filas.append({
        "Modelo": nombre,
        "Entrena (s)": round(segundos, 4),
        "Aciertos entrenamiento": modelo.score(X_train, y_train),
        "Aciertos prueba": modelo.score(X_test, y_test),
        "Precisión": precision_score(y_test, pred, zero_division=0),
        "Riesgo alto detectado (recall)": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    })
comparacion = pd.DataFrame(filas).set_index("Modelo")
comparacion.round(3)

**Cómo leer la tabla:** los aciertos totales engañan cuando la clase de interés es rara. Mire **recall** (de los casos de riesgo alto reales, cuántos detectó) y **precisión** (de los que marcó como riesgo alto, cuántos lo eran). **F1** resume ambos.

In [ ]:
comparacion[["Precisión", "Riesgo alto detectado (recall)", "F1"]].plot(
    kind="bar", figsize=(10, 5), color=["#4c78a8", "#e45756", "#54a24b"])
plt.title("Árbol, Naive Bayes y KNN sobre la misma partición")
plt.ylabel("Valor (más alto = mejor)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(GRAFICAS / "19_comparacion_modelos.png", dpi=150)
plt.show()

### 9.2 ¿Qué pasa si KNN no escala?
En la sesión 8 se vio que las variables con números grandes "secuestran" la distancia. Aquí `edad` e `imc` tienen escalas parecidas, pero `hombre`, `fuma` y `diabetes` valen 0, 1 o -1: veamos si importa.

In [ ]:
knn_crudo = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
pred_crudo = knn_crudo.predict(X_test)
knn_escalado = modelos["KNN (k=5)"]
print(f"KNN SIN escalar : aciertos {knn_crudo.score(X_test, y_test):.3f} | riesgo alto detectado {recall_score(y_test, pred_crudo):.3f}")
print(f"KNN escalado    : aciertos {knn_escalado.score(X_test, y_test):.3f} | riesgo alto detectado {recall_score(y_test, knn_escalado.predict(X_test)):.3f}")

### 9.3 El valor de *k*: entre memorizar y diluirse
Con *k* = 1 cada persona se juzga solo por su vecino más parecido (memoriza el entrenamiento). Con *k* grande el vecindario es tan amplio que el modelo tiende a responder siempre la clase mayoritaria. Se prueba una serie de valores impares (un *k* par puede empatar).

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
resultados_k = []
for k in [1, 3, 5, 11, 31, 51]:
    knn.set_params(kneighborsclassifier__n_neighbors=k).fit(X_train, y_train)
    pred_k = knn.predict(X_test)
    resultados_k.append({"k": k,
                         "Aciertos entrenamiento": knn.score(X_train, y_train),
                         "Aciertos prueba": knn.score(X_test, y_test),
                         "Riesgo alto detectado": recall_score(y_test, pred_k),
                         "F1": f1_score(y_test, pred_k)})
tabla_k = pd.DataFrame(resultados_k).set_index("k")
tabla_k.round(3)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(tabla_k.index, tabla_k["Aciertos entrenamiento"], marker="o", label="Entrenamiento")
plt.plot(tabla_k.index, tabla_k["Aciertos prueba"], marker="o", label="Prueba")
plt.plot(tabla_k.index, tabla_k["Riesgo alto detectado"], marker="s", linestyle="--", label="Riesgo alto detectado (prueba)")
plt.xscale("log")
plt.xticks(tabla_k.index, tabla_k.index)
plt.xlabel("k (cantidad de vecinos, escala logarítmica)")
plt.ylabel("Proporción")
plt.title("KNN: entre memorizar (k=1) y diluirse (k grande)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(GRAFICAS / "20_efecto_de_k.png", dpi=150)
plt.show()

### 9.4 ¿Funcionan igual en los tres conjuntos?
La misma comprobación que se le hizo al árbol en 8.9: porcentaje de riesgo alto detectado según el conjunto de origen.

In [ ]:
detectado = {}
for nombre, modelo in modelos.items():
    pred_m = pd.Series(modelo.predict(X_test), index=X_test.index)
    detectado[nombre] = {
        origen: recall_score(y_test[origen_prueba == origen], pred_m[origen_prueba == origen])
        for origen in ["adultos_salud", "insurance", "Medicalpremium"]}
pd.DataFrame(detectado).round(3)

### 9.5 Lo que dicen los resultados

*Para la lectura se usan las cifras de esta ejecución (`random_state=42`).*

- **Ningún modelo gana en todo.** El árbol detecta más casos de riesgo alto (61,5 %); KNN es el que mejor equilibra (F1 0,473 y precisión 0,786, pero detecta solo el 33,8 %); Naive Bayes, tal cual, casi no detecta nada (3,1 %).
- **Naive Bayes dependía del *a priori*.** Con `priors=[0.5, 0.5]` detecta el 67,7 %, pero a costa de muchas falsas alarmas (precisión 0,184 y aciertos totales de 0,708, por debajo de "decir siempre que no": 0,912). Se comprobó que el problema no eran los `-1` ni las variables 0/1: usando solo `edad` e `imc` el recall con el *a priori* normal fue incluso menor (1,5 %).
- **KNN sí necesita escalar:** sin escalar detecta el 6,2 % del riesgo alto; escalado, el 33,8 %.
- **El efecto de *k* se ve claro:** con *k* = 1 acierta el 99,9 % en entrenamiento y baja a 90,0 % en prueba (memoriza); con *k* = 31 y 51 el riesgo alto detectado cae a 20-23 %.
- **KNN heredó el mismo sesgo que el árbol:** en `adultos_salud` no detecta ningún caso de riesgo alto (0 %), mientras que en `insurance` detecta el 90,9 %. Aprende bien donde las 5 variables explican el riesgo y mal donde no.
- **Para TelecomUNO/este proyecto:** si importa **explicar** la decisión, el árbol; si importa **no dar falsas alarmas**, KNN; si importa **no dejar escapar casos** y se toleran falsas alarmas, Naive Bayes con *priors* balanceados.

**Cuidado:** la prueba tiene 740 personas y solo unas 65 son de riesgo alto, así que diferencias como F1 0,412 vs 0,473 son orientativas, no concluyentes.

> `python modelado_unificado.py` repite este torneo desde la consola y guarda la tabla en `data/resultados/`. El archivo `modelado.py` hace la comparación sobre `adultos_salud` completo (muestra de 40.000), con otro árbol: sirve de contraste.

In [ ]:
# Contraste: el mismo torneo, pero solo sobre adultos_salud (resultado guardado por modelado.py)
ruta_modelado = RAIZ / "data" / "resultados" / "comparacion_modelos.csv"
if ruta_modelado.exists():
    display(pd.read_csv(ruta_modelado))
else:
    print("Ejecute `python modelado.py` para generar", ruta_modelado.name)

## 10. Clustering: descubrir segmentos sin etiquetas

Hasta aquí todos los modelos tenían una respuesta que imitar (`HeartDisease` o `riesgo_alto`): **aprendizaje supervisado**. Ahora se **apaga la etiqueta**. Pregunta de negocio: *¿qué tipos de personas hay en `adultos_salud`, aunque nadie las haya clasificado nunca?*

| | Supervisado (partes 8-9) | No supervisado (hoy) |
|---|---|---|
| Datos | X e y | Solo X |
| Meta | Predecir la etiqueta | Descubrir grupos |
| Métrica | Aciertos, recall, F1 | Inercia, silueta y **utilidad real** |
| Riesgo | Sobreajuste | Interpretar grupos que no significan nada |

**Regla de oro:** `HeartDisease` **no entra** a X. Se usa *después*, solo para perfilar los grupos que el algoritmo encontró.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage

### 10.1 Preparar las variables
Se agrupa con 5 variables numéricas de estado de salud y edad (la edad se aproxima con el punto medio del rango, como en la parte 7). **Escalar es obligatorio:** K-means usa distancias, igual que KNN.

In [ ]:
segmentos = adultos_limpio.copy()
segmentos["edad"] = segmentos["AgeCategory"].map(punto_medio)
segmentos["enfermo"] = (segmentos["HeartDisease"] == "Yes").astype(int)

variables_cluster = ["edad", "BMI", "PhysicalHealth", "MentalHealth", "SleepTime"]
X_cluster = StandardScaler().fit_transform(segmentos[variables_cluster])
print("Personas:", X_cluster.shape[0], "| Variables:", variables_cluster)

### 10.2 Elegir *k*: método del codo y silueta
- **Inercia** (codo): suma de distancias al cuadrado de cada persona a su centro. Siempre baja al aumentar *k*; se busca dónde deja de valer la pena agregar grupos.
- **Silueta**: entre -1 y +1, mide qué tan cerca está cada persona de los suyos frente al grupo ajeno más próximo. Se calcula sobre una muestra de 10.000 personas porque su costo crece con el cuadrado de las filas.

In [ ]:
resultados_cluster = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_cluster)
    sil = silhouette_score(X_cluster, km.labels_, sample_size=10000, random_state=42)
    resultados_cluster.append({"k": k, "inercia": round(km.inertia_), "silueta": round(sil, 3)})
tabla_codo = pd.DataFrame(resultados_cluster).set_index("k")
tabla_codo

In [ ]:
K_ELEGIDO = 4

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
ejes[0].plot(tabla_codo.index, tabla_codo["inercia"], marker="o", linewidth=2.5)
ejes[0].set(xlabel="k (cantidad de grupos)", ylabel="Inercia (menos = grupos más compactos)", title="Método del codo")
ejes[1].plot(tabla_codo.index, tabla_codo["silueta"], marker="o", linewidth=2.5, color="#e45756")
ejes[1].axvline(K_ELEGIDO, color="gray", linestyle="--", label=f"k elegido = {K_ELEGIDO}")
ejes[1].set(xlabel="k (cantidad de grupos)", ylabel="Silueta promedio (más alto = mejor)", title="Coeficiente de silueta")
ejes[1].legend()
for eje in ejes:
    eje.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(GRAFICAS / "21_codo_y_silueta.png", dpi=150)
plt.show()

**Decisión sobre *k* (se justifica, no se esconde):**

- La silueta es **máxima en *k* = 2** (0,409), pero ese corte solo separa "buena salud" de "muchos días de mala salud" (7,6 % vs 16,3 % de enfermedad): demasiado grueso para actuar.
- De *k* = 3 a *k* = 8 la silueta es casi igual (0,23 a 0,26) y **el codo no es nítido**: la curva baja de forma suave.
- Se elige ***k* = 4**: da grupos distintos por edad y estado de salud, con riesgo muy diferente, y son pocos para operar cuatro acciones distintas.

Siluetas de 0,25 son normales en datos reales: las personas no vienen en islas separadas. Lo importante es que los grupos sean **útiles**.

### 10.3 K-means final y perfilamiento
Se renumeran los grupos de menor a mayor edad media: así la numeración no depende del azar de la semilla. El perfil usa **medias reales** (años, días), no los centros estandarizados.

In [ ]:
km_final = KMeans(n_clusters=K_ELEGIDO, n_init=10, random_state=42).fit(X_cluster)
segmentos["cluster"] = km_final.labels_

orden = segmentos.groupby("cluster")["edad"].mean().sort_values().index
segmentos["cluster"] = segmentos["cluster"].map({viejo: nuevo for nuevo, viejo in enumerate(orden)})

perfil = segmentos.groupby("cluster")[variables_cluster].mean().round(1)
perfil.insert(0, "personas", segmentos["cluster"].value_counts().sort_index())
perfil.insert(1, "% del total", (perfil["personas"] / len(segmentos) * 100).round(1))
perfil["% con enfermedad"] = (segmentos.groupby("cluster")["enfermo"].mean() * 100).round(1)
print(f"Promedio general de enfermedad cardíaca: {segmentos['enfermo'].mean() * 100:.1f} %")
perfil

Variables que **no** entraron al clustering: sirven para comprobar que los grupos significan algo.

In [ ]:
verificacion = pd.DataFrame({
    "% fuma": (segmentos["Smoking"] == "Yes").groupby(segmentos["cluster"]).mean() * 100,
    "% dificultad para caminar": (segmentos["DiffWalking"] == "Yes").groupby(segmentos["cluster"]).mean() * 100,
    "% diabetes": segmentos["Diabetic"].str.startswith("Yes").groupby(segmentos["cluster"]).mean() * 100,
}).round(1)
verificacion

### 10.4 Ponerles nombre
El entregable del clustering **no es el número de grupo**: es el nombre, el retrato y la acción asociada. Los nombres dependen de estos datos y de esta numeración; si cambia algo, hay que revisarlos.

In [ ]:
nombres = {0: "Jóvenes sanos",
           1: "Adultos con malestar emocional",
           2: "Mayores con mala salud física",
           3: "Mayores estables"}
segmentos["segmento"] = segmentos["cluster"].map(nombres)

# Mapa de calor: qué distingue a cada grupo (centros estandarizados, en el orden nuevo)
centros = pd.DataFrame(km_final.cluster_centers_, columns=variables_cluster)
centros.index = [orden.get_loc(i) for i in range(K_ELEGIDO)]
centros = centros.sort_index()

plt.figure(figsize=(10, 4.5))
plt.imshow(centros.values, cmap="coolwarm", vmin=-2, vmax=2, aspect="auto")
plt.colorbar(label="Desviaciones estándar sobre el promedio")
plt.xticks(range(len(variables_cluster)), variables_cluster, rotation=20)
plt.yticks(range(K_ELEGIDO), [f"{i} · {nombres[i]}" for i in centros.index])
for i in range(K_ELEGIDO):
    for j in range(len(variables_cluster)):
        plt.text(j, i, f"{centros.values[i, j]:+.1f}", ha="center", va="center")
plt.title("Qué distingue a cada grupo (rojo = más alto que el promedio)")
plt.tight_layout()
plt.savefig(GRAFICAS / "22_perfiles_mapa_calor.png", dpi=150)
plt.show()

In [ ]:
por_segmento = segmentos.groupby("segmento")["enfermo"].mean().mul(100).reindex(nombres.values())

plt.figure(figsize=(9, 5))
barras = plt.bar(por_segmento.index.str.replace(" con ", "\ncon ").str.replace("Mayores ", "Mayores\n"),
                 por_segmento.values, color="#e45756")
plt.axhline(segmentos["enfermo"].mean() * 100, color="gray", linestyle="--", label="Promedio general")
for barra, valor in zip(barras, por_segmento.values):
    plt.text(barra.get_x() + barra.get_width() / 2, valor + 0.2, f"{valor:.1f} %", ha="center")
plt.ylabel("% con enfermedad cardíaca")
plt.title("La enfermedad no se usó para agrupar, pero separa a los grupos")
plt.legend()
plt.tight_layout()
plt.savefig(GRAFICAS / "23_enfermedad_por_grupo.png", dpi=150)
plt.show()

| Segmento | Retrato | Enfermedad | Acción sugerida |
|---|---|---|---|
| **Jóvenes sanos** (35 %) | ~37 años, buena salud física y mental | 1,7 % | Prevención general y hábitos (un tercio fuma). |
| **Adultos con malestar emocional** (9 %) | ~45 años, ~23 días de mala salud mental al mes | 6,9 % | Programas de salud mental; vigilar cardiovascular. |
| **Mayores con mala salud física** (9 %) | ~62 años, ~27 días de mala salud física, el 59 % con dificultad para caminar, el 30 % con diabetes | **23,0 %** | **Prioridad clínica:** seguimiento cercano y control de diabetes. |
| **Mayores estables** (47 %) | ~68 años, buena salud declarada | 12,3 % | Chequeos periódicos: el riesgo sube por edad aunque se sientan bien. |

Los grupos coinciden con lo esperable (más edad y peor salud, más enfermedad) **sin que el algoritmo haya visto la enfermedad**: eso es una buena señal de que los segmentos no son numerología. Pero ojo: *es asociación, no causa*.

Los grupos también se pueden ver en 2 dimensiones con PCA (reduce las 5 variables a 2 ejes):

In [ ]:
muestra_pca = segmentos.sample(8000, random_state=42)
pca = PCA(n_components=2, random_state=42)
puntos = pca.fit_transform(StandardScaler().fit_transform(muestra_pca[variables_cluster]))

plt.figure(figsize=(8, 6))
for g in range(K_ELEGIDO):
    m = (muestra_pca["cluster"] == g).to_numpy()
    plt.scatter(puntos[m, 0], puntos[m, 1], s=6, alpha=0.5, label=nombres[g])
plt.xlabel(f"Componente 1 ({pca.explained_variance_ratio_[0]:.0%} de la variación)")
plt.ylabel(f"Componente 2 ({pca.explained_variance_ratio_[1]:.0%} de la variación)")
plt.title("Los grupos en 2 dimensiones (muestra de 8.000 personas)")
plt.legend(markerscale=3)
plt.tight_layout()
plt.savefig(GRAFICAS / "24_grupos_pca.png", dpi=150)
plt.show()

### 10.5 Contraste con el clustering jerárquico
El método aglomerativo empieza con cada persona como un grupo y va **fundiendo** los dos más parecidos; el registro de las fusiones es el **dendrograma**. Es costoso (compara todos los pares), por eso se usa una **muestra de 60 personas**. Con `linkage="ward"` se funden los grupos que menos aumentan la varianza interna.

In [ ]:
mini = segmentos[variables_cluster].sample(n=60, random_state=42)
Z = linkage(StandardScaler().fit_transform(mini), method="ward")
print("Primeras fusiones (distancia):", Z[:5, 2].round(1))
print("Últimas 4 fusiones (distancia):", Z[-4:, 2].round(1))
for k in (2, 3, 4, 5):
    tamanos = np.bincount(fcluster(Z, t=k, criterion="maxclust"))[1:]
    print(f"  corte en {k} grupos -> tamaños {sorted(tamanos.tolist(), reverse=True)}")

corte = (Z[-K_ELEGIDO, 2] + Z[-(K_ELEGIDO - 1), 2]) / 2
plt.figure(figsize=(13, 5))
dendrogram(Z, color_threshold=corte, no_labels=True)
plt.axhline(corte, color="gray", linestyle="--", label=f"Corte en {K_ELEGIDO} ramas")
plt.ylabel("Distancia (Ward)")
plt.title("Dendrograma de 60 personas de adultos_salud")
plt.legend()
plt.tight_layout()
plt.savefig(GRAFICAS / "25_dendrograma.png", dpi=150)
plt.show()

**Lectura:** las primeras fusiones ocurren a distancias casi 0 (hay personas prácticamente idénticas) y las últimas a 7,5, 7,6, 9,6 y 10,9. Un corte en 4 ramas es **compatible** con el *k* = 4 de K-means, pero los saltos entre fusiones son moderados: con solo 60 personas el dendrograma **apoya** la elección, no la demuestra.

### 10.6 Límites que hay que confesar
- **K-means siempre devuelve grupos**, aunque no existan: el mérito está en que sean distintos y útiles, no en que el algoritmo los haya encontrado.
- **Silueta baja (~0,25) y sin codo nítido:** *k* es una decisión razonada, no un hallazgo.
- `PhysicalHealth` y `MentalHealth` valen casi siempre 0 o 30 días; eso condiciona los grupos (por eso *k* = 2 separa justo esas variables).
- Asume grupos redondeados y es sensible a valores extremos (que aquí se conservaron a propósito).
- El clustering podría **enriquecer** el modelo de la parte 8: la columna `cluster` como variable nueva de entrada. No se probó aquí.

> `python clustering.py` repite todo desde la consola y guarda tablas en `data/resultados/` y gráficas en `graficas/clustering/`.

### 10.7 El mismo análisis con `insurance`
Misma receta, otros datos: 500.000 personas con `age`, `bmi`, `children` y `smoker` (convertida a 0/1). **`charges` (el costo) queda fuera** del clustering y se usa después para perfilar, igual que `HeartDisease` antes.

In [ ]:
seguros = insurance.copy()
seguros["fuma"] = (seguros["smoker"] == "yes").astype(int)
variables_seg = ["age", "bmi", "children", "fuma"]
X_seg = StandardScaler().fit_transform(seguros[variables_seg])

resultados_seg = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_seg)
    sil = silhouette_score(X_seg, km.labels_, sample_size=10000, random_state=42)
    resultados_seg.append({"k": k, "inercia": round(km.inertia_), "silueta": round(sil, 3)})
pd.DataFrame(resultados_seg).set_index("k")

**Decisión sobre *k*:** la silueta es máxima en *k* = 2 (0,345), pero ese corte solo separa fumadores de no fumadores, algo que el EDA ya mostraba. *k* = 4 tiene la **segunda mejor silueta (0,302)** y conserva a los fumadores como un grupo mientras parte a los no fumadores por edad e hijos: ahí está lo nuevo.

**Dato importante:** los fumadores forman **un grupo propio con cualquier *k***, porque al escalar una variable 0/1 pesa mucho en la distancia. Es a la vez un hallazgo (fumar domina) y una limitación (el clustering no logra partir a los fumadores).

In [ ]:
K_SEG = 4
km_seg = KMeans(n_clusters=K_SEG, n_init=10, random_state=42).fit(X_seg)
seguros["cluster"] = km_seg.labels_
orden_seg = seguros.groupby("cluster")["age"].mean().sort_values().index
seguros["cluster"] = seguros["cluster"].map({viejo: nuevo for nuevo, viejo in enumerate(orden_seg)})

nombres_seg = {0: "Jóvenes no fumadores", 1: "Fumadores", 2: "Familias con hijos", 3: "No fumadores maduros"}
perfil_seg = seguros.groupby("cluster")[variables_seg].mean().round(2)
perfil_seg.insert(0, "segmento", perfil_seg.index.map(nombres_seg))
perfil_seg.insert(1, "personas", seguros["cluster"].value_counts().sort_index())
perfil_seg["costo medio"] = seguros.groupby("cluster")["charges"].mean().round(0)
print(f"Costo medio general: {seguros['charges'].mean():,.0f}")
perfil_seg

In [ ]:
costo_seg = perfil_seg.set_index("segmento")["costo medio"]

plt.figure(figsize=(9, 5))
barras = plt.bar(costo_seg.index.str.replace(" no ", "\nno ").str.replace(" con ", "\ncon "),
                 costo_seg.values, color="#e45756")
plt.axhline(seguros["charges"].mean(), color="gray", linestyle="--", label="Promedio general")
for barra, valor in zip(barras, costo_seg.values):
    plt.text(barra.get_x() + barra.get_width() / 2, valor * 1.01, f"{valor:,.0f}", ha="center")
plt.ylabel("Costo médico medio (charges)")
plt.title("El costo no se usó para agrupar, pero separa a los grupos")
plt.legend()
plt.tight_layout()
plt.savefig(GRAFICAS / "26_costo_por_grupo_insurance.png", dpi=150)
plt.show()

| Segmento | Personas | Retrato | Costo medio |
|---|---|---|---|
| **Jóvenes no fumadores** | 145.264 (29 %) | ~26 años, pocos hijos (0,5) | 4.201 |
| **Fumadores** | 102.862 (21 %) | ~38 años, el 58 % hombres | **32.230** |
| **Familias con hijos** | 114.332 (23 %) | ~41 años, **2,7 hijos** en promedio | 9.451 |
| **No fumadores maduros** | 137.542 (28 %) | ~53 años, pocos hijos | 11.683 |

**Lectura:** el costo medio de los fumadores es ~7,7 veces el de los jóvenes no fumadores, y eso confirma el hallazgo del EDA. Lo que aporta el clustering es el **orden dentro de los no fumadores**: el costo sube con la edad y las familias con hijos quedan en un punto intermedio. El IMC casi no distingue grupos (promedios entre 29 y 32), coherente con que la obesidad solo encarece cuando se combina con fumar.

*Con `python clustering.py insurance` se repite todo, incluidos el PCA y el dendrograma (la muestra de 60 registros separa a los fumadores como una rama aparte, a una distancia alta).*

## 11. Conclusiones

1. **Unir tres conjuntos distintos exige decisiones.** Se usaron solo las 5 variables compartidas (`edad`, `imc`, `hombre`, `fuma`, `diabetes`), se creó un objetivo común (`riesgo_alto`, el 9 % más alto en cada conjunto) y se tomó una muestra de 986 filas de cada uno para que ninguno domine.
2. **Cita rangos, no un solo número.** Con la muestra de semilla 42 el árbol único detecta el 61,5 % del riesgo alto (84,6 % de aciertos, 31 % de precisión), pero al repetir con 20 muestras distintas (parte 8.11) el promedio es **75 %** (entre 57 % y 85 %), con una precisión de **26 %** (entre 19 % y 41 %) y **77 %** de aciertos. Detecta bastante riesgo alto, pero con muchas falsas alarmas.
3. **No funciona igual en los tres conjuntos.** En promedio de las 20 repeticiones detecta el 96 % del riesgo alto en `insurance`, el 74 % en `adultos_salud` (entre 41 % y 100 %) y el 54 % en `Medicalpremium` (entre 32 % y 73 %). El árbol aprendió sobre todo la regla de `insurance` (IMC alto y fumar), probablemente porque ahí las variables disponibles explican muy bien el costo.
4. **Variables más usadas:** IMC (30 %), edad (29 %), diabetes (23 %) y fumar (17 %). El sexo casi no pesa.
5. **El `-1` revela el conjunto de origen.** Como cada conjunto no tiene alguna variable, el árbol puede deducir de dónde viene una fila (por ejemplo, sin dato de diabetes = `insurance`) y usar reglas distintas para cada uno. Es una limitación de unir conjuntos con columnas diferentes.

6. **Naive Bayes y KNN frente al árbol (parte 9):** sobre la misma partición (semilla 42), el árbol detecta más riesgo alto (61,5 %), KNN equilibra mejor (F1 0,473, precisión 0,786) y Naive Bayes solo funciona si se corrige su *a priori* (recall de 3,1 % a 67,7 %). KNN sin escalar cae a 6,2 % de detección. Ninguno gana en todo: la elección depende de si importa explicar, evitar falsas alarmas o no dejar escapar casos.
7. **Clustering (parte 10):** en `adultos_salud` K-means con *k* = 4 encontró cuatro segmentos (Jóvenes sanos, Adultos con malestar emocional, Mayores con mala salud física y Mayores estables) cuya tasa de enfermedad va de 1,7 % a 23,0 % sin haber usado la enfermedad para agrupar. La silueta es baja (~0,25) y *k* se justificó, no se descubrió. Con `insurance` K-means (*k* = 4) reencuentra que los fumadores son un grupo aparte (costo medio 32.230 frente a 4.201 a 11.683 en los demás) y ordena a los no fumadores por edad e hijos.

8. **Otros modelos en igualdad de condiciones.** Con validación cruzada sobre `adultos_salud` (`modelado_justo.py`), la regresión logística separa mejor a los enfermos que el árbol (ROC AUC 0,83 frente a 0,72). Para predecir costos (`modelado_regresion.py`), el Random Forest es el mejor en `insurance` y `Medicalpremium`, y el árbol supera a la regresión lineal.

**Limitaciones:** el riesgo alto significa cosas distintas en cada conjunto (enfermedad vs. costo vs. prima), la edad de `adultos_salud` es aproximada (punto medio del rango) y la prueba tiene solo 740 personas (por eso se da un rango) y la profundidad 4 se eligió con una sola muestra.

**Próximos pasos:** probar con más filas (subiendo `n` en la parte 7.6), probar el `cluster` como variable de entrada del modelo de la parte 8 y conectar el procesamiento por bloques con `main.py`.